# Quick Start

Minimum usage for `run_with_cache` with a monitoring engine.


In [1]:
import torch
from transformers import AutoTokenizer
from transformers.models.gpt2_p.modeling_gpt2 import HookedGPT2Model

from monitoring import MonitoringEngine
from monitoring.config import CaptureSchedule, HookSelection, MonitoringConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
token = tokenizer.encode("The future of AI is", return_tensors="pt").to(device)



/home/nengneng/miniconda3/envs/proj-dmx/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Init Monitoring Engine with Configuration

### CaptureSchedule

Control when to capture:

```python
CaptureSchedule(
    step_stride=5,           # capture every 5th token
    step_offset=0,           # start from token 0
    request_stride=1,        # capture every request
    capture_prefill=True,    # capture prefill phase
    capture_decode=True,     # capture decode phase
)
```

### HookSelection

Control which hooks to enable:

```python
# Predefined modes
HookSelection(mode='full')       # all 363 hooks
HookSelection(mode='attention')  # attention-related hooks only
HookSelection(mode='mlp')        # MLP-related hooks only

# Custom selection
HookSelection(mode='custom', custom_hooks=['blocks.0.attn.hook_q', 'blocks.0.attn.hook_k'])
```


In [2]:
config = MonitoringConfig(
    hooks=HookSelection(mode="full"),
    schedule=CaptureSchedule(
        step_stride=1,
        step_offset=0,
        warmup_steps=0,
        capture_prefill=True,
        capture_decode=True,
        request_stride=1,
        request_offset=0,
        warmup_requests=0,
    ),
)

engine = MonitoringEngine(async_enabled=device.type == "cuda", config=config)

## Load Hooked Model and Config

In [3]:
hf_hooked_model = HookedGPT2Model.from_pretrained(
    "gpt2",
    attn_implementation="eager",
    torch_dtype=torch.float32,
)
hf_hooked_model.to(device)
hf_hooked_model.eval()
hf_hooked_model.monitoring_engine = engine

init_ms = engine.prepare_for_model(hf_hooked_model)

`torch_dtype` is deprecated! Use `dtype` instead!


In [4]:
engine.begin_request(0)
engine.start_step(phase="prefill")
outputs, cache_dict = hf_hooked_model.run_with_cache(
    token,
    use_cache=True,
    past_key_values=None,
    output_hidden_states=True,
    output_attentions=True,
    return_dict=True,
)

engine.end_step()

## decode the next token
next_token = token[:, -1:]
engine.start_step(phase="decode")
outputs, cache_dict = hf_hooked_model.run_with_cache(
    next_token,
    use_cache=True,
    past_key_values=outputs.past_key_values,
    output_hidden_states=True,
    output_attentions=True,
    return_dict=True,
)
engine.end_step()


Get Tensors ready from backend workers

In [5]:
count = 0
for name, fut in cache_dict.items():
    if fut is None:
        continue
    cache_dict[name] = fut.result()
    if cache_dict[name] is not None:
        count += 1

assert count == len(cache_dict.items()), "Some are missing"
print("All tensor are ready")

        

All tensor are ready


In [6]:
cache_dict.keys()
print(cache_dict['blocks.0.hook_resid_pre'].shape)

torch.Size([1, 1, 768])


In [7]:
engine.clear_completed_results()
cache_dict.clear()


## Generate with Monitoring (CUDA + ClickHouse pipeline)

Requires: CUDA, native backend built, ClickHouse running, and dmx_host installed.


### 1) Environment variables and CUDA check
Set native backend flags and ensure CUDA is available.


In [8]:
import os
import torch

os.environ.setdefault("MON_NATIVE_TO_CPU", "1")
os.environ.setdefault("MON_NATIVE_CALLBACK", "1")
os.environ.setdefault("MON_NATIVE_BUILDER", "1")
os.environ.setdefault("MON_NATIVE_BATCH", "0")
os.environ.setdefault("MON_NATIVE_AUTOCLEAR", "0")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA required; skip this section if not available.")


### 2) ClickHouse connection config
Override with environment variables if needed.


In [9]:
try:
    import dmx_host.clickhouse_client as clickhouse_client
except Exception:
    import clickhouse_client  # type: ignore

db_cfg = clickhouse_client.ClickHouseClientConfig()
db_cfg.host = os.environ.get("DMX_DB_HOST", "localhost")
db_cfg.port = int(os.environ.get("DMX_DB_PORT", "9000"))
db_cfg.username = os.environ.get("DMX_DB_USER", "default")
db_cfg.password = os.environ.get("DMX_DB_PASSWORD", "")
db_cfg.database = os.environ.get("DMX_DB_DATABASE", "default")
db_cfg.table = os.environ.get("DMX_DB_TABLE", "offload")
db_cfg.create_database_if_missing = True


### 3) HostEngine pipeline
Stage 1 parses tensors; Stage 2 inserts into ClickHouse.


In [10]:
from monitoring import HostEngineConfig
from dmx_host.engine import EngineConfig, QueueConfig, StageConfig
import dmx_host.dmx_interface as dmx_interface

stage_one = StageConfig(
    name="stage_one",
    parallelism=1,
    process_fn=dmx_interface.stage_one_parsing_and_wait,
    thread_init_config=None,
    thread_init=dmx_interface.stage_one_thread_init,
    thread_cleanup=dmx_interface.stage_one_thread_cleanup,
    input_queue=QueueConfig(1, None, None, None, None, 400, None),
)
stage_two = StageConfig(
    name="stage_two",
    parallelism=1,
    process_fn=clickhouse_client.clickhouse_insert,
    thread_init_config=db_cfg,
    thread_init=clickhouse_client.clickhouse_init,
    thread_cleanup=clickhouse_client.clickhouse_cleanup,
    input_queue=QueueConfig(1, None, None, None, None, 400, None),
)
host_cfg = HostEngineConfig(
    stages=[stage_one, stage_two],
    input_handler=dmx_interface.input_handler_v1,
    engine_config=EngineConfig(),
)


### 4) Monitoring config
Use full hook capture for this demo.


In [11]:
from monitoring.config import CaptureSchedule, HookSelection, MonitoringConfig

cfg = MonitoringConfig(
    hooks=HookSelection(mode="full"),
    schedule=CaptureSchedule(capture_prefill=True, capture_decode=True),
)


### 5) Load model and bind monitoring engine


In [12]:
from transformers import AutoTokenizer
from transformers.models.gpt2_p.modeling_gpt2 import HookedGPT2LMHeadModel
from monitoring import MonitoringEngine
from monitoring.generate import generate_with_monitoring

model_id = "gpt2"
engine = MonitoringEngine(
    async_enabled=True,
    config=cfg,
    model_id=model_id,
    db_config=host_cfg,
)

model = HookedGPT2LMHeadModel.from_pretrained(
    model_id,
    attn_implementation="eager",
    torch_dtype=torch.float32,
).cuda().eval()
model.monitoring_engine = engine
engine.prepare_for_model(model)

tokenizer = AutoTokenizer.from_pretrained(model_id)


DATABASE INIT DONE


### 6) Inference and DB insert


In [13]:
encoded = tokenizer("The future of AI is", return_tensors="pt")
input_ids = encoded["input_ids"].cuda()
attention_mask = encoded["attention_mask"].cuda()

with torch.no_grad():
    output_ids = generate_with_monitoring(
        model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=8,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))
engine.close()


The future of AI is uncertain. The future of AI is uncertain
